# 4. Through-the-door testing and policy evaluation

This notebook creates and scores the simulated 2025 through-the-door cohort, compares cut-offs with the chronological holdout, reviews calibration, evaluates the three-tier decision policy, and stress-tests underwriter accuracy, illustrative costs and DTI overrides.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from sklearn.metrics import brier_score_loss, roc_auc_score

project_root = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "src" / "origination_scorecard" / "generator.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project")
sys.path.insert(0, str(project_root / "src"))

from origination_scorecard.generator import GeneratorConfig, _make_applications, _make_performance

kigb_dir = project_root / "data" / "KIGB Modelling"
ttd_dir = project_root / "data" / "TTD Modelling"
ttd_dir.mkdir(parents=True, exist_ok=True)

## Generate application-time inputs and keep outcomes separate

In [2]:
sample_size = 2_000
random_seed = 20_261_002
config = GeneratorConfig(n_accounts=sample_size, random_seed=random_seed)
rng = np.random.default_rng(random_seed)
applications = _make_applications(config, rng)
_, truth = _make_performance(applications, config, rng)

date_rng = np.random.default_rng(random_seed + 1)
applications["application_date"] = (
    pd.Timestamp("2025-01-01")
    + pd.to_timedelta(date_rng.integers(0, 181, sample_size), unit="D")
)
applications["account_id"] = [f"RETRO-2025-{i:06d}" for i in range(1, sample_size + 1)]

application_columns = [
    "account_id", "application_date", "requested_loan_amount", "disposable_income",
    "debt_to_income_ratio", "recent_application_count", "revolving_utilisation",
    "historic_defaults", "time_at_address_months", "loan_purpose",
]
applicants = applications[application_columns].copy()
outcomes = applications[["master_account_key", "account_id"]].merge(
    truth[["master_account_key", "total_missed_payments"]],
    on="master_account_key", validate="one_to_one",
)
missed = outcomes["total_missed_payments"]
outcomes["observed_group"] = np.select(
    [missed.eq(0), missed.between(1, 2), missed.ge(3)],
    ["Known good", "Intermediate", "Known bad"],
    default="Invalid",
)
outcomes["observed_bad"] = missed.ge(3).astype(int)

post_outcome_fields = {"observed_bad", "observed_group", "total_missed_payments", "kigb_bad", "target_bad"}
if post_outcome_fields & set(applicants.columns):
    raise AssertionError("A post-outcome field entered the scoring inputs")

print(f"TTD applicants: {len(applicants):,}")
print(f"Application dates: {applicants.application_date.min().date()} to {applicants.application_date.max().date()}")
display(outcomes["observed_group"].value_counts().rename_axis("Observed group").reset_index(name="Accounts"))

TTD applicants: 2,000
Application dates: 2025-01-01 to 2025-06-30


,Observed group,Accounts
0,Known good,1880
1,Known bad,80
2,Intermediate,40


## Fit on KIGB development data and score TTD applicants

In [3]:
kigb_train = pd.read_csv(kigb_dir / "KIGB in sample.csv")
kigb_holdout = pd.read_csv(kigb_dir / "KIGB out sample.csv")

scaled_predictors = [
    "debt_to_income_ratio", "recent_application_count", "revolving_utilisation",
    "historic_defaults", "log_address_tenure",
]
numeric_inputs = [
    "debt_to_income_ratio", "recent_application_count", "revolving_utilisation",
    "historic_defaults", "disposable_income", "time_at_address_months",
]

def prepare_inputs(data, target):
    rows = data.copy()
    for column in numeric_inputs:
        rows[column] = pd.to_numeric(rows[column], errors="raise")
    required = [target, "loan_purpose", *numeric_inputs]
    if rows[required].isna().any().any():
        raise ValueError("Missing model input or target")
    if not np.isfinite(rows[numeric_inputs].to_numpy()).all():
        raise ValueError("Infinite model input")
    if rows["time_at_address_months"].lt(0).any():
        raise ValueError("Negative address tenure")
    rows["low_disposable_income_flag"] = (rows["disposable_income"] < 150).astype(float)
    rows["log_address_tenure"] = np.log1p(rows["time_at_address_months"])
    return rows

def make_design(rows, means, stds):
    x = (rows[scaled_predictors] - means) / stds
    x.insert(0, "const", 1.0)
    x["low_disposable_income_flag"] = rows["low_disposable_income_flag"]
    x["loan_purpose=education"] = (rows["loan_purpose"] == "education").astype(float)
    return x.astype(float)

factor = 20 / np.log(2)
offset = 600 - factor * np.log(50)

def pd_to_score(probability):
    p = np.clip(np.asarray(probability, dtype=float), 1e-12, 1 - 1e-12)
    return offset + factor * np.log((1 - p) / p)

kigb_train = prepare_inputs(kigb_train, "kigb_bad")
applicants_for_score = applicants.copy()
applicants_for_score["kigb_bad"] = 0  # placeholder required only by input validation; never used for fitting/evaluation
applicants_for_score = prepare_inputs(applicants_for_score, "kigb_bad")

means = kigb_train[scaled_predictors].mean()
stds = kigb_train[scaled_predictors].std(ddof=0)
x_train = make_design(kigb_train, means, stds)
x_ttd = make_design(applicants_for_score, means, stds)[x_train.columns]
y_train = kigb_train["kigb_bad"].astype(int)

model = sm.Logit(y_train, x_train).fit(maxiter=200, disp=False)
if not model.mle_retvals.get("converged", False):
    raise RuntimeError("KIGB model did not converge")

evaluation = applicants.copy()
evaluation["predicted_pd"] = np.asarray(model.predict(x_ttd))
evaluation["score"] = pd_to_score(evaluation["predicted_pd"])
cutoff_score = float(pd_to_score([0.025])[0])
evaluation["decision"] = np.where(
    evaluation["score"] >= cutoff_score,
    "Accept",
    "Flag / decline",
)
evaluation = evaluation.merge(
    outcomes[["account_id", "total_missed_payments", "observed_group", "observed_bad"]],
    on="account_id", how="left", validate="one_to_one",
)
if evaluation["observed_bad"].isna().any():
    raise ValueError("A TTD outcome failed to link")

output_path = ttd_dir / "TTD retrospective scored clean.csv"
evaluation["application_date"] = pd.to_datetime(evaluation["application_date"]).dt.strftime("%Y-%m-%d")
evaluation.to_csv(output_path, index=False, lineterminator="\n")
print(f"Saved {len(evaluation):,} scored TTD rows to {output_path}")

Saved 2,000 scored TTD rows to C:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\TTD Modelling\TTD retrospective scored clean.csv


## Overall TTD discrimination and calibration

In [4]:
auc = roc_auc_score(evaluation["observed_bad"], evaluation["predicted_pd"])
overall = pd.DataFrame([{
    "Accounts": len(evaluation),
    "Expected bads": evaluation["predicted_pd"].sum(),
    "Observed bads": evaluation["observed_bad"].sum(),
    "Predicted bad rate": evaluation["predicted_pd"].mean(),
    "Observed bad rate": evaluation["observed_bad"].mean(),
    "Calibration gap": evaluation["observed_bad"].mean() - evaluation["predicted_pd"].mean(),
    "Brier score": brier_score_loss(evaluation["observed_bad"], evaluation["predicted_pd"]),
    "ROC AUC": auc,
    "Gini": 2 * auc - 1,
}])
display(overall.style.format({
    "Accounts": "{:,.0f}", "Expected bads": "{:.1f}", "Observed bads": "{:,.0f}",
    "Predicted bad rate": "{:.2%}", "Observed bad rate": "{:.2%}",
    "Calibration gap": "{:+.2%}", "Brier score": "{:.4f}", "ROC AUC": "{:.4f}", "Gini": "{:.4f}",
}))

,Accounts,Expected bads,Observed bads,Predicted bad rate,Observed bad rate,Calibration gap,Brier score,ROC AUC,Gini
0,"2,000",77.1,80,3.85%,4.00%,+0.15%,0.0383,0.5887,0.1774


## Cut-off robustness: chronological holdout versus TTD

In [5]:
cutoffs = [0.0200, 0.0225, 0.0250, 0.0300, 0.0350]

def cutoff_metrics(sample_name, rows, outcome, cutoff):
    accepted = rows.loc[rows["predicted_pd"] < cutoff]
    flagged = rows.loc[rows["predicted_pd"] >= cutoff]
    return {
        "Sample": sample_name, "PD cutoff": cutoff,
        "Minimum score": float(pd_to_score([cutoff])[0]),
        "Accepted accounts": len(accepted), "Approval rate": len(accepted) / len(rows),
        "Predicted accepted bad rate": accepted["predicted_pd"].mean(),
        "Observed accepted bad rate": accepted[outcome].mean(),
        "Calibration gap": accepted[outcome].mean() - accepted["predicted_pd"].mean(),
        "Bad capture in flagged group": flagged[outcome].sum() / rows[outcome].sum(),
    }

robustness = pd.DataFrame([
    cutoff_metrics(name, rows, outcome, cutoff)
    for name, rows, outcome in [
        ("Chronological holdout", kigb_holdout, "kigb_bad"),
        ("Simulated TTD", evaluation, "observed_bad"),
    ]
    for cutoff in cutoffs
])
display(robustness.style.format({
    "PD cutoff": "{:.2%}", "Minimum score": "{:.1f}", "Approval rate": "{:.1%}",
    "Predicted accepted bad rate": "{:.2%}", "Observed accepted bad rate": "{:.2%}",
    "Calibration gap": "{:+.2%}", "Bad capture in flagged group": "{:.1%}",
}))

,Sample,PD cutoff,Minimum score,Accepted accounts,Approval rate,Predicted accepted bad rate,Observed accepted bad rate,Calibration gap,Bad capture in flagged group
0,Chronological holdout,2.00%,599.4,158,6.3%,1.42%,1.90%,+0.48%,97.1%
1,Chronological holdout,2.25%,595.9,282,11.2%,1.73%,1.42%,-0.31%,96.1%
2,Chronological holdout,2.50%,592.8,459,18.3%,1.98%,1.53%,-0.46%,93.1%
3,Chronological holdout,3.00%,587.4,889,35.4%,2.36%,2.25%,-0.11%,80.4%
4,Chronological holdout,3.50%,582.8,1323,52.8%,2.65%,2.57%,-0.08%,66.7%
5,Simulated TTD,2.00%,599.4,103,5.1%,1.37%,2.91%,+1.55%,96.2%
6,Simulated TTD,2.25%,595.9,208,10.4%,1.77%,3.37%,+1.60%,91.2%
7,Simulated TTD,2.50%,592.8,364,18.2%,2.03%,2.75%,+0.71%,87.5%
8,Simulated TTD,3.00%,587.4,751,37.5%,2.40%,3.33%,+0.93%,68.8%
9,Simulated TTD,3.50%,582.8,1082,54.1%,2.66%,3.33%,+0.67%,55.0%


## TTD calibration around the policy boundaries

In [6]:
band_edges = [0, 0.015, 0.020, 0.025, 0.030, 0.035, 0.050, 0.075, 0.10, 1.000001]
labels = [
    "<1.50%", "1.50–1.99%", "2.00–2.49%", "2.50–2.99%", "3.00–3.49%",
    "3.50–4.99%", "5.00–7.49%", "7.50–9.99%", "10.00%+",
]
banded = evaluation.copy()
banded["PD band"] = pd.cut(banded["predicted_pd"], band_edges, labels=labels, include_lowest=True, right=False)
ttd_calibration = banded.groupby("PD band", observed=False).agg(
    Accounts=("observed_bad", "size"), Observed_bads=("observed_bad", "sum"),
    Mean_predicted_PD=("predicted_pd", "mean"), Observed_bad_rate=("observed_bad", "mean"),
    Mean_score=("score", "mean"),
).reset_index()
ttd_calibration["Calibration gap"] = ttd_calibration["Observed_bad_rate"] - ttd_calibration["Mean_predicted_PD"]
display(ttd_calibration.style.format({
    "Accounts": "{:,.0f}", "Observed_bads": "{:,.0f}", "Mean_predicted_PD": "{:.2%}",
    "Observed_bad_rate": "{:.2%}", "Calibration gap": "{:+.2%}", "Mean_score": "{:.1f}",
}, na_rep="—"))

,PD band,Accounts,Observed_bads,Mean_predicted_PD,Observed_bad_rate,Mean_score,Calibration gap
0,<1.50%,65,2,1.09%,3.08%,617.8,+1.99%
1,1.50–1.99%,38,1,1.84%,2.63%,601.9,+0.79%
2,2.00–2.49%,261,7,2.29%,2.68%,595.4,+0.39%
3,2.50–2.99%,387,15,2.75%,3.88%,590.1,+1.13%
4,3.00–3.49%,331,11,3.24%,3.32%,585.1,+0.08%
5,3.50–4.99%,548,20,4.12%,3.65%,578.1,-0.47%
6,5.00–7.49%,271,17,5.96%,6.27%,566.9,+0.31%
7,7.50–9.99%,63,4,8.46%,6.35%,555.9,-2.12%
8,10.00%+,36,3,11.83%,8.33%,545.4,-3.50%


## Limits on interpretation

The TTD sample changes the random seed and application dates, but it still comes from the same synthetic generator and embeds the same assumed relationships between inputs and outcomes. It does not test new economic conditions, a different applicant population, operational data quality, selection bias from a live lending policy, fairness, fraud, or real loss severity.

Before using the scorecard for lending decisions, it would need genuine application-time data linked to subsequently observed outcomes, an untouched out-of-time validation sample, calibration and stability monitoring, segment and fairness analysis, independent model validation, and policy economics based on actual exposure, loss given default, review cost and the value of declined good customers.

# Policy economics and robustness testing

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

project_root = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data" / "TTD Modelling" / "TTD retrospective scored clean.csv").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the scored TTD sample")

ttd_path = project_root / "data" / "TTD Modelling" / "TTD retrospective scored clean.csv"
policy_sample = pd.read_csv(ttd_path)
policy_sample["observed_bad"] = pd.to_numeric(policy_sample["observed_bad"], errors="raise").astype(int)
policy_sample["predicted_pd"] = pd.to_numeric(policy_sample["predicted_pd"], errors="raise")
policy_sample["debt_to_income_ratio"] = pd.to_numeric(policy_sample["debt_to_income_ratio"], errors="raise")

AUTO_ACCEPT_PD = 0.0250
AUTO_DECLINE_PD = 0.0350
policy_sample["policy_band"] = np.select(
    [
        policy_sample["predicted_pd"] < AUTO_ACCEPT_PD,
        policy_sample["predicted_pd"].between(AUTO_ACCEPT_PD, AUTO_DECLINE_PD, inclusive="both"),
        policy_sample["predicted_pd"] > AUTO_DECLINE_PD,
    ],
    ["1. Auto accept", "2. Human review", "3. Auto decline"],
    default="Unassigned",
)
if policy_sample["policy_band"].eq("Unassigned").any():
    raise ValueError("A TTD applicant was not assigned to a policy band")
print(f"Loaded {len(policy_sample):,} TTD applicants")

Loaded 2,000 TTD applicants


## Tier performance and referral workload

In [8]:
total_bads = policy_sample["observed_bad"].sum()
tier_table = policy_sample.groupby("policy_band", observed=True).agg(
    Accounts=("observed_bad", "size"),
    Expected_bads=("predicted_pd", "sum"),
    Observed_bads=("observed_bad", "sum"),
    Predicted_bad_rate=("predicted_pd", "mean"),
    Observed_bad_rate=("observed_bad", "mean"),
).reset_index()
tier_table["Applicant share"] = tier_table["Accounts"] / len(policy_sample)
tier_table["Share of all observed bads"] = tier_table["Observed_bads"] / total_bads
tier_table["Calibration gap"] = tier_table["Observed_bad_rate"] - tier_table["Predicted_bad_rate"]
display(tier_table.style.format({
    "Accounts": "{:,.0f}", "Expected_bads": "{:.1f}", "Observed_bads": "{:,.0f}",
    "Applicant share": "{:.1%}", "Share of all observed bads": "{:.1%}",
    "Predicted_bad_rate": "{:.2%}", "Observed_bad_rate": "{:.2%}", "Calibration gap": "{:+.2%}",
}))

review_count = int(policy_sample["policy_band"].eq("2. Human review").sum())
print(f"Referral workload: {review_count:,} applicants ({review_count / len(policy_sample):.1%} of TTD).")

,policy_band,Accounts,Expected_bads,Observed_bads,Predicted_bad_rate,Observed_bad_rate,Applicant share,Share of all observed bads,Calibration gap
0,1. Auto accept,364,7.4,10,2.03%,2.75%,18.2%,12.5%,+0.71%
1,2. Human review,718,21.4,26,2.98%,3.62%,35.9%,32.5%,+0.64%
2,3. Auto decline,918,48.3,44,5.26%,4.79%,45.9%,55.0%,-0.47%


Referral workload: 718 applicants (35.9% of TTD).


## Deterministic policy economics

In [9]:
# Illustrative, configurable cost units.
COST_ACCEPTED_BAD = 100.0
COST_REVIEW = 2.0
COST_DECLINED_GOOD = 10.0

def evaluate_policy(data, review_accuracy, dti_override=None,
                    accepted_bad_cost=COST_ACCEPTED_BAD,
                    review_cost=COST_REVIEW,
                    declined_good_cost=COST_DECLINED_GOOD):
    band = data["policy_band"].copy()
    override = pd.Series(False, index=data.index)
    if dti_override is not None:
        override = band.eq("1. Auto accept") & data["debt_to_income_ratio"].gt(dti_override)
        band.loc[override] = "2. Human review"

    auto_accept = band.eq("1. Auto accept")
    review = band.eq("2. Human review")
    auto_decline = band.eq("3. Auto decline")
    bad = data["observed_bad"].eq(1)
    good = ~bad

    auto_accept_bads = int((auto_accept & bad).sum())
    review_bads = int((review & bad).sum())
    review_goods = int((review & good).sum())
    auto_decline_goods = int((auto_decline & good).sum())

    expected_accepted_bads = auto_accept_bads + review_bads * (1 - review_accuracy)
    expected_accepted_goods = int((auto_accept & good).sum()) + review_goods * review_accuracy
    expected_declined_goods = auto_decline_goods + review_goods * (1 - review_accuracy)
    expected_accepted = expected_accepted_bads + expected_accepted_goods
    total_cost = (
        expected_accepted_bads * accepted_bad_cost
        + int(review.sum()) * review_cost
        + expected_declined_goods * declined_good_cost
    )

    return {
        "Review accuracy": review_accuracy,
        "DTI override": "None" if dti_override is None else f"> {dti_override:.0%}",
        "Applicants reviewed": int(review.sum()),
        "Review rate": review.mean(),
        "Incremental reviews from DTI": int(override.sum()),
        "Expected accepted accounts": expected_accepted,
        "Expected approval rate": expected_accepted / len(data),
        "Expected accepted bads": expected_accepted_bads,
        "Expected accepted bad rate": expected_accepted_bads / expected_accepted,
        "Expected good declines": expected_declined_goods,
        "Expected bads prevented": data["observed_bad"].sum() - expected_accepted_bads,
        "Accepted-bad cost": expected_accepted_bads * accepted_bad_cost,
        "Review cost": int(review.sum()) * review_cost,
        "Good-decline cost": expected_declined_goods * declined_good_cost,
        "Total scenario cost": total_cost,
    }

base_policy = pd.DataFrame([
    evaluate_policy(policy_sample, accuracy)
    for accuracy in [0.80, 0.90, 0.98, 1.00]
])
display(base_policy.style.format({
    "Review accuracy": "{:.0%}", "Review rate": "{:.1%}",
    "Expected accepted accounts": "{:,.1f}", "Expected approval rate": "{:.1%}",
    "Expected accepted bads": "{:.1f}", "Expected accepted bad rate": "{:.2%}",
    "Expected good declines": "{:.1f}", "Expected bads prevented": "{:.1f}",
    "Accepted-bad cost": "{:,.1f}", "Review cost": "{:,.1f}",
    "Good-decline cost": "{:,.1f}", "Total scenario cost": "{:,.1f}",
}))

,Review accuracy,DTI override,Applicants reviewed,Review rate,Incremental reviews from DTI,Expected accepted accounts,Expected approval rate,Expected accepted bads,Expected accepted bad rate,Expected good declines,Expected bads prevented,Accepted-bad cost,Review cost,Good-decline cost,Total scenario cost
0,80%,None,718,35.9%,0,922.8,46.1%,15.2,1.65%,1012.4,64.8,"1,520.0","1,436.0","10,124.0","13,080.0"
1,90%,None,718,35.9%,0,989.4,49.5%,12.6,1.27%,943.2,67.4,"1,260.0","1,436.0","9,432.0","12,128.0"
2,98%,None,718,35.9%,0,"1,042.7",52.1%,10.5,1.01%,887.8,69.5,"1,052.0","1,436.0","8,878.4","11,366.4"
3,100%,None,718,35.9%,0,"1,056.0",52.8%,10.0,0.95%,874.0,70.0,"1,000.0","1,436.0","8,740.0","11,176.0"


## Candidate DTI overrides

Each candidate rule sends an applicant who would otherwise be auto-accepted to review when DTI exceeds the stated threshold. The first table shows the observed concentration of bads in those incremental referrals. The second combines the rule with different underwriter accuracies.

In [10]:
auto_accept_mask = policy_sample["policy_band"].eq("1. Auto accept")
override_rows = []
for threshold in [0.15, 0.20, 0.25]:
    referred = policy_sample.loc[auto_accept_mask & policy_sample["debt_to_income_ratio"].gt(threshold)]
    bads = int(referred["observed_bad"].sum())
    override_rows.append({
        "DTI threshold": threshold,
        "Incremental reviews": len(referred),
        "Observed bads in incremental reviews": bads,
        "Observed bad rate": referred["observed_bad"].mean(),
        "Bads per 100 incremental reviews": 100 * bads / len(referred) if len(referred) else np.nan,
        "Share of auto-accepted bads reached": bads / policy_sample.loc[auto_accept_mask, "observed_bad"].sum(),
    })
override_effectiveness = pd.DataFrame(override_rows)
display(override_effectiveness.style.format({
    "DTI threshold": "{:.0%}", "Observed bad rate": "{:.2%}",
    "Bads per 100 incremental reviews": "{:.2f}", "Share of auto-accepted bads reached": "{:.1%}",
}))

stress_test = pd.DataFrame([
    evaluate_policy(policy_sample, accuracy, threshold)
    for accuracy in [0.80, 0.90, 0.98]
    for threshold in [None, 0.25, 0.20, 0.15]
])
display(stress_test.style.format({
    "Review accuracy": "{:.0%}", "Review rate": "{:.1%}",
    "Expected accepted accounts": "{:,.1f}", "Expected approval rate": "{:.1%}",
    "Expected accepted bads": "{:.1f}", "Expected accepted bad rate": "{:.2%}",
    "Expected good declines": "{:.1f}", "Expected bads prevented": "{:.1f}",
    "Accepted-bad cost": "{:,.1f}", "Review cost": "{:,.1f}",
    "Good-decline cost": "{:,.1f}", "Total scenario cost": "{:,.1f}",
}))

,DTI threshold,Incremental reviews,Observed bads in incremental reviews,Observed bad rate,Bads per 100 incremental reviews,Share of auto-accepted bads reached
0,15%,64,4,6.25%,6.25,40.0%
1,20%,32,1,3.12%,3.12,10.0%
2,25%,17,1,5.88%,5.88,10.0%


,Review accuracy,DTI override,Applicants reviewed,Review rate,Incremental reviews from DTI,Expected accepted accounts,Expected approval rate,Expected accepted bads,Expected accepted bad rate,Expected good declines,Expected bads prevented,Accepted-bad cost,Review cost,Good-decline cost,Total scenario cost
0,80%,None,718,35.9%,0,922.8,46.1%,15.2,1.65%,1012.4,64.8,"1,520.0","1,436.0","10,124.0","13,080.0"
1,80%,> 25%,735,36.8%,17,918.8,45.9%,14.4,1.57%,1015.6,65.6,"1,440.0","1,470.0","10,156.0","13,066.0"
2,80%,> 20%,750,37.5%,32,915.8,45.8%,14.4,1.57%,1018.6,65.6,"1,440.0","1,500.0","10,186.0","13,126.0"
3,80%,> 15%,782,39.1%,64,907.6,45.4%,12.0,1.32%,1024.4,68.0,"1,200.0","1,564.0","10,244.0","13,008.0"
4,90%,None,718,35.9%,0,989.4,49.5%,12.6,1.27%,943.2,67.4,"1,260.0","1,436.0","9,432.0","12,128.0"
5,90%,> 25%,735,36.8%,17,986.9,49.3%,11.7,1.19%,944.8,68.3,"1,170.0","1,470.0","9,448.0","12,088.0"
6,90%,> 20%,750,37.5%,32,985.4,49.3%,11.7,1.19%,946.3,68.3,"1,170.0","1,500.0","9,463.0","12,133.0"
7,90%,> 15%,782,39.1%,64,979.8,49.0%,9.0,0.92%,949.2,71.0,900.0,"1,564.0","9,492.0","11,956.0"
8,98%,None,718,35.9%,0,"1,042.7",52.1%,10.5,1.01%,887.8,69.5,"1,052.0","1,436.0","8,878.4","11,366.4"
9,98%,> 25%,735,36.8%,17,"1,041.4",52.1%,9.5,0.92%,888.2,70.5,954.0,"1,470.0","8,881.6","11,305.6"


## Cost sensitivity: mistaken acceptance versus review and decline

In [11]:
cost_sensitivity = pd.DataFrame([
    evaluate_policy(
        policy_sample,
        review_accuracy=accuracy,
        dti_override=threshold,
        accepted_bad_cost=bad_cost,
        review_cost=COST_REVIEW,
        declined_good_cost=COST_DECLINED_GOOD,
    ) | {"Assumed cost per accepted bad": bad_cost}
    for bad_cost in [25.0, 50.0, 100.0, 200.0]
    for accuracy in [0.90, 0.98]
    for threshold in [None, 0.20, 0.15]
])
display(cost_sensitivity[[
    "Assumed cost per accepted bad", "Review accuracy", "DTI override",
    "Expected accepted bads", "Expected good declines", "Applicants reviewed",
    "Accepted-bad cost", "Review cost", "Good-decline cost", "Total scenario cost",
]].style.format({
    "Review accuracy": "{:.0%}", "Expected accepted bads": "{:.1f}",
    "Expected good declines": "{:.1f}", "Accepted-bad cost": "{:,.1f}",
    "Review cost": "{:,.1f}", "Good-decline cost": "{:,.1f}", "Total scenario cost": "{:,.1f}",
}))

,Assumed cost per accepted bad,Review accuracy,DTI override,Expected accepted bads,Expected good declines,Applicants reviewed,Accepted-bad cost,Review cost,Good-decline cost,Total scenario cost
0,25.000000,90%,None,12.6,943.2,718,315.0,"1,436.0","9,432.0","11,183.0"
1,25.000000,90%,> 20%,11.7,946.3,750,292.5,"1,500.0","9,463.0","11,255.5"
2,25.000000,90%,> 15%,9.0,949.2,782,225.0,"1,564.0","9,492.0","11,281.0"
3,25.000000,98%,None,10.5,887.8,718,263.0,"1,436.0","8,878.4","10,577.4"
4,25.000000,98%,> 20%,9.5,888.5,750,238.5,"1,500.0","8,884.6","10,623.1"
5,25.000000,98%,> 15%,6.6,889.0,782,165.0,"1,564.0","8,890.4","10,619.4"
6,50.000000,90%,None,12.6,943.2,718,630.0,"1,436.0","9,432.0","11,498.0"
7,50.000000,90%,> 20%,11.7,946.3,750,585.0,"1,500.0","9,463.0","11,548.0"
8,50.000000,90%,> 15%,9.0,949.2,782,450.0,"1,564.0","9,492.0","11,506.0"
9,50.000000,98%,None,10.5,887.8,718,526.0,"1,436.0","8,878.4","10,840.4"


## Interpretation boundary

These tables make the trade-offs visible, but they do not prove that 2.50%, 3.50% or any DTI rule is commercially optimal. The TTD outcomes and the underwriter decisions are simulated. A real policy needs observed losses, exposure and recovery data, actual review outcomes and costs, the value of declined good customers, capacity constraints, fairness testing and independent validation. The thresholds should then be selected on development data and confirmed on a genuinely untouched later cohort.